# MP03 Financial Services *Pipeline*
**Role:** Financial Services Pipeline Lead | Team 05
**Course:** CIS 3120, Baruch College

In [1]:
!pip install anthropic folium beautifulsoup4 requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 11.6 MB/s eta 0:00:00


In [2]:
import json, re, time
from datetime import date, timedelta
import requests
from bs4 import BeautifulSoup
import folium
import pandas as pd
from anthropic import Anthropic

USER_AGENT       = "CIS3120 MP03 Team 05 - carmen.li2@baruch.cuny.edu"
REQUEST_HEADERS  = {"User-Agent": USER_AGENT}
EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
NOMINATIM_URL    = "https://nominatim.openstreetmap.org/search"
EDGAR_PAUSE      = 0.15
NOMINATIM_PAUSE  = 1.10
MODEL_ID         = "claude-haiku-4-5-20251001"
print("Setup complete.")

Setup complete.


In [3]:
from google.colab import userdata
client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
print("Anthropic client ready.")

Anthropic client ready.


## Canonical Pipeline

In [4]:
def search_edgar_one_phrase(
    phrase: str,
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
) -> tuple[list[dict], int]:
    """Query EDGAR full-text search for one phrase across a date window."""
    all_hits: list[dict] = []
    total = 0
    for page in range(max_pages):
        params = {
            "q": phrase, "dateRange": "custom",
            "startdt": start_date.isoformat(), "enddt": end_date.isoformat(),
            "forms": forms, "from": page * 100,
        }
        response = requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        hits = data.get("hits", {}).get("hits", [])
        all_hits.extend(hits)
        total = data.get("hits", {}).get("total", {}).get("value", 0)
        if (page + 1) * 100 >= total:
            break
        time.sleep(EDGAR_PAUSE)
    return all_hits, total


def search_edgar_all_phrases(
    phrases: list[str],
    start_date: date,
    end_date: date,
    forms: str = "8-K",
    max_pages: int = 2,
    max_filings: int = 500,
) -> list[dict]:
    """Run search_edgar_one_phrase across all phrases with dedup and backoff."""
    seen: set[str] = set()
    deduped: list[dict] = []
    backoff_waits = [5, 10, 15]
    for phrase in phrases:
        attempts = 0
        while attempts <= len(backoff_waits):
            try:
                hits, _ = search_edgar_one_phrase(phrase, start_date, end_date, forms, max_pages)
                break
            except requests.RequestException as exc:
                if attempts == len(backoff_waits):
                    print(f"  WARNING: {phrase!r} failed after retries; skipping")
                    hits = []
                    break
                time.sleep(backoff_waits[attempts])
                attempts += 1
        for hit in hits:
            key = hit.get("_id", "")
            if key and key not in seen:
                seen.add(key)
                deduped.append(hit)
            if len(deduped) >= max_filings:
                return deduped
        time.sleep(EDGAR_PAUSE)
    return deduped


def build_exhibit_url(hit: dict) -> str:
    """Construct the SEC archive URL for the exhibit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}"


def fetch_exhibit_text(hit: dict, max_chars: int = 8000) -> tuple[str, str]:
    """Fetch and HTML-strip the exhibit text. Returns (text, url)."""
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"
    return text, url


EXTRACTION_SYSTEM_PROMPT = """You are an analyst reviewing 8-K filing exhibits to identify announcements of location-related corporate events: openings, closings, relocations, or expansions of physical facilities (stores, warehouses, distribution centers, offices, plants).

Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (e.g., "NY", "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null

Be strict. If the filing mentions a location only in passing (e.g., headquarters address in the boilerplate), return is_location_event: false. Return only the JSON object with no preamble, no markdown fences, no explanation."""


def extract_with_claude(filing: dict) -> dict:
    """Classify and extract structured location data from a single filing."""
    response = client.messages.create(
        model=MODEL_ID, max_tokens=300,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": filing["text"]}],
    )
    raw = response.content[0].text.strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"is_location_event": False, "_parse_error": raw[:200]}
    record = {**filing, **parsed}
    record["input_tokens"]  = response.usage.input_tokens
    record["output_tokens"] = response.usage.output_tokens
    return record


def geocode_location(city: str, state: str | None) -> tuple[float, float] | None:
    """Geocode a US city/state pair via Nominatim."""
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1, "countrycodes": "us"}
    response = requests.get(NOMINATIM_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()
    data = response.json()
    time.sleep(NOMINATIM_PAUSE)
    if not data:
        return None
    return float(data[0]["lat"]), float(data[0]["lon"])


EVENT_COLORS = {
    "opening": "green", "closing": "red",
    "relocation": "orange", "expansion": "blue", "other": "gray",
}
US_CENTER_LAT, US_CENTER_LON = 39.8, -98.6
print("All canonical functions defined.")

All canonical functions defined.


## 3. Financial Services Ticker and Phrase Lists

**Key change from v1:** We now embed company names directly in the search phrases.
This bypasses the broken tickers field in EDGAR entirely and guarantees matches.

In [5]:
FINANCIAL_SERVICES_TICKERS = [
    # Money-center banks (seed)
    "JPM", "BAC", "WFC", "C",
    # Regional banks (seed)
    "PNC", "USB", "TFC",
    # Asset management (seed)
    "BLK", "BX",
    # Insurance (seed)
    "MET", "PRU",
    # Payments (seed)
    "V", "MA", "AXP",
    # Capital markets — added
    "GS", "MS", "SCHW",
    # Fintech — added
    "PYPL", "SQ",
]

# Company-name embedded phrases — guarantees EDGAR returns these companies' filings
# This is the key fix: instead of searching generically and filtering by ticker,
# we search for company name + location keyword together.
FINANCIAL_SERVICES_PHRASES = [
    # Seed phrases (kept for broad recall)
    '"new branch"',
    '"branch opening"',
    '"branch closure"',
    '"branch closing"',
    '"branch consolidation"',
    '"regional office"',
    '"office closure"',
    '"operations center"',
    '"data center"',
    '"new location"',
    # Extended phrases
    '"new office"',
    '"office opening"',
    '"trading floor"',
    '"advisory office"',
]

# Company-name targeted phrases for direct EDGAR hits
# These are searched separately and merged with the above
FINANCIAL_SERVICES_COMPANY_PHRASES = [
    '"JPMorgan" "branch"',
    '"Wells Fargo" "branch"',
    '"Bank of America" "branch"',
    '"Citibank" "branch"',
    '"Truist" "branch"',
    '"PNC" "branch"',
    '"U.S. Bank" "branch"',
    '"Goldman Sachs" "office"',
    '"Morgan Stanley" "office"',
    '"Charles Schwab" "branch"',
    '"American Express" "office"',
    '"Mastercard" "office"',
    '"MetLife" "office"',
    '"Prudential" "office"',
    '"BlackRock" "office"',
    '"Blackstone" "office"',
]

# Combined list used by the pipeline
ALL_FS_PHRASES = FINANCIAL_SERVICES_PHRASES + FINANCIAL_SERVICES_COMPANY_PHRASES

print(f"Tickers: {len(FINANCIAL_SERVICES_TICKERS)}")
print(f"Generic phrases: {len(FINANCIAL_SERVICES_PHRASES)}")
print(f"Company-targeted phrases: {len(FINANCIAL_SERVICES_COMPANY_PHRASES)}")
print(f"Total phrases: {len(ALL_FS_PHRASES)}")

Tickers: 19
Generic phrases: 14
Company-targeted phrases: 16
Total phrases: 30


In [13]:
def filter_candidates_by_tickers(
    candidates: list[dict],
    ticker_list: list[str],
) -> list[dict]:
    """Restrict candidates to actual Financial Services companies only.

    Excludes mortgage trusts, SPVs, and other subsidiary entities that
    share parent company names but file unrelated 8-Ks.
    """
    target_tickers = {t.upper() for t in ticker_list}

    # Exact company name matching — must match the parent company
    # Exclusion keywords filter out mortgage trusts and SPVs
    EXCLUDE_KEYWORDS = [
        "mortgage trust", "commercial mortgage", "real estate finance trust",
        "pass-through", "securities trust", "asset trust", "funding trust",
        "master trust", "issuance trust", "card trust", "auto trust",
    ]

    ticker_name_map = {
        "JPM":  ["jpmorgan chase & co", "jpmorgan chase bank"],
        "BAC":  ["bank of america corporation", "bank of america, n.a"],
        "WFC":  ["wells fargo & company", "wells fargo bank, n.a"],
        "C":    ["citigroup inc", "citibank, n.a"],
        "PNC":  ["pnc financial services", "pnc bank, n.a"],
        "USB":  ["u.s. bancorp", "united states bancorp"],
        "TFC":  ["truist financial", "truist bank"],
        "BLK":  ["blackrock, inc", "blackrock inc"],
        "BX":   ["blackstone inc", "blackstone group"],
        "MET":  ["metlife, inc", "metlife insurance"],
        "PRU":  ["prudential financial"],
        "V":    ["visa inc"],
        "MA":   ["mastercard incorporated", "mastercard international"],
        "AXP":  ["american express company"],
        "GS":   ["goldman sachs group", "goldman sachs & co"],
        "MS":   ["morgan stanley"],
        "SCHW": ["charles schwab corporation", "charles schwab bank"],
        "PYPL": ["paypal holdings"],
        "SQ":   ["block, inc", "square, inc"],
    }

    filtered = []
    for hit in candidates:
        src = hit.get("_source", {})
        display = " ".join(src.get("display_names") or []).lower()

        # Skip SPVs and mortgage trusts immediately
        if any(excl in display for excl in EXCLUDE_KEYWORDS):
            continue

        # Method 1: tickers field
        hit_tickers = {t.upper() for t in (src.get("tickers") or [])}
        if hit_tickers & target_tickers:
            filtered.append(hit)
            continue

        # Method 2: exact parent company name matching
        matched = False
        for ticker, fragments in ticker_name_map.items():
            if ticker in target_tickers:
                for fragment in fragments:
                    if fragment in display:
                        filtered.append(hit)
                        matched = True
                        break
            if matched:
                break

    return filtered

print("Filter updated — SPVs and mortgage trusts excluded.")

Filter updated — SPVs and mortgage trusts excluded.


In [7]:
def run_industry_pipeline(
    industry_label: str,
    ticker_list: list[str],
    phrase_list: list[str],
    window_days: int,
) -> tuple[list[dict], int, float]:
    """Run all five pipeline stages for one industry slice.

    Returns (geocoded_events, candidate_count, estimated_cost_usd).
    The candidate_count and cost are always returned even if 0 events found,
    so the window-results table is always populated correctly.

    Note: returns a 3-tuple instead of just a list so trial stats are
    never lost when Claude finds 0 location events.
    """
    print(f"\n{'='*60}")
    print(f"Pipeline: {industry_label} | window={window_days} days")
    print(f"{'='*60}")

    end_date   = date.today()
    start_date = end_date - timedelta(days=window_days)
    print(f"Date window: {start_date} to {end_date}")

    # Stage 1
    print(f"\nStage 1: Searching {len(phrase_list)} phrases...")
    all_candidates = search_edgar_all_phrases(phrase_list, start_date, end_date)
    print(f"  Raw candidates: {len(all_candidates)}")

    filtered = filter_candidates_by_tickers(all_candidates, ticker_list)
    candidate_count = len(filtered)
    print(f"  Filtered candidates: {candidate_count}")

    if not filtered:
        print("  No candidates after filtering.")
        return [], 0, 0.0

    # Stage 2 + 3
    print(f"\nStage 2+3: Fetching and classifying {candidate_count} candidates...")
    location_events = []
    total_input_tokens  = 0
    total_output_tokens = 0
    fetch_failures = 0

    for i, hit in enumerate(filtered):
        try:
            text, url = fetch_exhibit_text(hit)
        except Exception as exc:
            fetch_failures += 1
            print(f"  [warn] Fetch failed: {exc}")
            time.sleep(EDGAR_PAUSE)
            continue

        src = hit.get("_source", {})
        filing = {
            "company":   (src.get("display_names") or ["(unknown)"])[0],
            "ticker":    (src.get("tickers") or [None])[0],
            "file_date": src.get("file_date"),
            "accession": hit["_id"].split(":")[0],
            "url": url, "text": text,
        }

        try:
            record = extract_with_claude(filing)
            total_input_tokens  += record.get("input_tokens", 0)
            total_output_tokens += record.get("output_tokens", 0)
        except Exception as exc:
            print(f"  [warn] Extraction failed for {filing['company']}: {exc}")
            continue

        if record.get("is_location_event"):
            location_events.append(record)

        time.sleep(EDGAR_PAUSE)
        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/{candidate_count} ({len(location_events)} events so far)")

    estimated_cost = (
        (total_input_tokens  / 1_000_000) * 1.0 +
        (total_output_tokens / 1_000_000) * 5.0
    )
    print(f"\n  Location events: {len(location_events)} | cost: ${estimated_cost:.4f}")

    # Stage 4
    print(f"\nStage 4: Geocoding {len(location_events)} events...")
    geocoded = []
    for event in location_events:
        coords = geocode_location(event.get("city"), event.get("state"))
        if coords:
            event["latitude"]  = coords[0]
            event["longitude"] = coords[1]
            event["industry"]  = industry_label
            geocoded.append(event)
        else:
            print(f"  [warn] Could not geocode: {event.get('city')}, {event.get('state')}")

    print(f"\nDone. Geocoded: {len(geocoded)} | candidates: {candidate_count} | cost: ${estimated_cost:.4f}")
    return geocoded, candidate_count, estimated_cost


def summarize_window_trial(
    industry_label: str,
    window_days: int,
    candidate_count: int,
    event_count: int,
    estimated_cost_usd: float,
) -> dict:
    """Record the result of one window-tuning trial.

    Returns a dict with the five exact columns required by the assignment spec.
    """
    return {
        "industry":           industry_label,
        "window_days":        window_days,
        "candidate_count":    candidate_count,
        "event_count":        event_count,
        "estimated_cost_usd": round(estimated_cost_usd, 4),
    }

print("run_industry_pipeline and summarize_window_trial defined.")

run_industry_pipeline and summarize_window_trial defined.


In [8]:
window_results  = pd.DataFrame(columns=["industry","window_days","candidate_count","event_count","estimated_cost_usd"])
cumulative_cost = 0.0
COST_CEILING    = 3.00
fs_events       = {}  # stores events by window: fs_events[30], fs_events[60], etc.
print(f"Ready. Cost ceiling: ${COST_CEILING}")

Ready. Cost ceiling: $3.0


In [9]:
# 30-day window
# Uses ALL_FS_PHRASES (generic + company-targeted)
events_30, candidates_30, cost_30 = run_industry_pipeline(
    "Financial Services", FINANCIAL_SERVICES_TICKERS, ALL_FS_PHRASES, window_days=30
)
cumulative_cost += cost_30
fs_events[30] = events_30

row = summarize_window_trial("Financial Services", 30, candidates_30, len(events_30), cost_30)
window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)
print(f"\nCumulative cost: ${cumulative_cost:.4f} / ${COST_CEILING}")
display(window_results)


Pipeline: Financial Services | window=30 days
Date window: 2026-04-11 to 2026-05-11

Stage 1: Searching 30 phrases...
  Raw candidates: 500
  Filtered candidates: 4

Stage 2+3: Fetching and classifying 4 candidates...

  Location events: 0 | cost: $0.0111

Stage 4: Geocoding 0 events...

Done. Geocoded: 0 | candidates: 4 | cost: $0.0111

Cumulative cost: $0.0111 / $3.0


/tmp/ipykernel_24866/3353550782.py:10: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111


In [10]:
# 60-day window
# Only run if 30-day gave < 8 events
if len(events_30) >= 8:
    print(f"Target reached at 30 days ({len(events_30)} events). Skip this cell.")
else:
    events_60, candidates_60, cost_60 = run_industry_pipeline(
        "Financial Services", FINANCIAL_SERVICES_TICKERS, ALL_FS_PHRASES, window_days=60
    )
    cumulative_cost += cost_60
    fs_events[60] = events_60
    row = summarize_window_trial("Financial Services", 60, candidates_60, len(events_60), cost_60)
    window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)
    print(f"\nCumulative cost: ${cumulative_cost:.4f} / ${COST_CEILING}")
    display(window_results)


Pipeline: Financial Services | window=60 days
Date window: 2026-03-12 to 2026-05-11

Stage 1: Searching 30 phrases...
  Raw candidates: 500
  Filtered candidates: 8

Stage 2+3: Fetching and classifying 8 candidates...

  Location events: 0 | cost: $0.0230

Stage 4: Geocoding 0 events...

Done. Geocoded: 0 | candidates: 8 | cost: $0.0230

Cumulative cost: $0.0341 / $3.0


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111
1,Financial Services,60,8,0,0.0230


In [11]:
# 90-day window
best_so_far = max(len(v) for v in fs_events.values()) if fs_events else 0
if best_so_far >= 8:
    print(f"Target reached ({best_so_far} events). Skip this cell.")
else:
    events_90, candidates_90, cost_90 = run_industry_pipeline(
        "Financial Services", FINANCIAL_SERVICES_TICKERS, ALL_FS_PHRASES, window_days=90
    )
    cumulative_cost += cost_90
    fs_events[90] = events_90
    row = summarize_window_trial("Financial Services", 90, candidates_90, len(events_90), cost_90)
    window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)
    print(f"\nCumulative cost: ${cumulative_cost:.4f} / ${COST_CEILING}")
    display(window_results)


Pipeline: Financial Services | window=90 days
Date window: 2026-02-10 to 2026-05-11

Stage 1: Searching 30 phrases...
  Raw candidates: 500
  Filtered candidates: 12

Stage 2+3: Fetching and classifying 12 candidates...
  Processed 10/12 (0 events so far)

  Location events: 0 | cost: $0.0351

Stage 4: Geocoding 0 events...

Done. Geocoded: 0 | candidates: 12 | cost: $0.0351

Cumulative cost: $0.0692 / $3.0


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111
1,Financial Services,60,8,0,0.0230
2,Financial Services,90,12,0,0.0351


In [16]:
best_so_far = max(len(v) for v in fs_events.values()) if fs_events else 0
if best_so_far >= 8:
    print(f"Target reached ({best_so_far} events). Skip this cell.")
else:
    events_180, candidates_180, cost_180 = run_industry_pipeline(
        "Financial Services", FINANCIAL_SERVICES_TICKERS, ALL_FS_PHRASES, window_days=180
    )
    cumulative_cost += cost_180
    fs_events[180] = events_180
    row = summarize_window_trial("Financial Services", 180, candidates_180, len(events_180), cost_180)
    window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)
    print(f"\nCumulative cost: ${cumulative_cost:.4f} / ${COST_CEILING}")
    display(window_results)


Pipeline: Financial Services | window=180 days
Date window: 2025-11-12 to 2026-05-11

Stage 1: Searching 30 phrases...
  Raw candidates: 500
  Filtered candidates: 0
  No candidates after filtering.

Cumulative cost: $0.0692 / $3.0


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111
1,Financial Services,60,8,0,0.0230
2,Financial Services,90,12,0,0.0351
3,Financial Services,180,0,0,0.0000


In [17]:
best_so_far = max(len(v) for v in fs_events.values()) if fs_events else 0
if best_so_far >= 8:
    print(f"Target reached ({best_so_far} events). Skip this cell.")
else:
    events_360, candidates_360, cost_360 = run_industry_pipeline(
        "Financial Services", FINANCIAL_SERVICES_TICKERS, ALL_FS_PHRASES, window_days=360
    )
    cumulative_cost += cost_360
    fs_events[360] = events_360
    row = summarize_window_trial("Financial Services", 360, candidates_360, len(events_360), cost_360)
    window_results = pd.concat([window_results, pd.DataFrame([row])], ignore_index=True)
    print(f"\nCumulative cost: ${cumulative_cost:.4f} / ${COST_CEILING}")
    display(window_results)


Pipeline: Financial Services | window=360 days
Date window: 2025-05-16 to 2026-05-11

Stage 1: Searching 30 phrases...
  Raw candidates: 500
  Filtered candidates: 1

Stage 2+3: Fetching and classifying 1 candidates...

  Location events: 1 | cost: $0.0029

Stage 4: Geocoding 1 events...
  [warn] Could not geocode: None, None

Done. Geocoded: 0 | candidates: 1 | cost: $0.0029

Cumulative cost: $0.0721 / $3.0


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111
1,Financial Services,60,8,0,0.0230
2,Financial Services,90,12,0,0.0351
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0029


In [18]:
print("=" * 60)
print("WINDOW-TUNING RESULTS — Financial Services")
print("=" * 60)
display(window_results)
print(f"\nTotal cost: ${cumulative_cost:.4f} / ${COST_CEILING}")

best_window = None
best_events = []
for w in [30, 60, 90, 180, 360]:
    if w in fs_events and len(fs_events[w]) >= 8:
        best_window = w
        best_events = fs_events[w]
        break
if best_window is None and fs_events:
    best_window = max(fs_events, key=lambda w: len(fs_events[w]))
    best_events = fs_events[best_window]

print(f"\nBest window: {best_window} days | Events: {len(best_events)}")

WINDOW-TUNING RESULTS — Financial Services


,industry,window_days,candidate_count,event_count,estimated_cost_usd
0,Financial Services,30,4,0,0.0111
1,Financial Services,60,8,0,0.0230
2,Financial Services,90,12,0,0.0351
3,Financial Services,180,0,0,0.0000
4,Financial Services,360,1,0,0.0029



Total cost: $0.0721 / $3.0

Best window: 30 days | Events: 0


In [19]:
# ── Inspect events ─────────────────────────────────────────────────────────
print(f"Reviewing {len(best_events)} events from {best_window}-day window:\n")
for i, event in enumerate(best_events):
    print(f"[{i+1}] {event['file_date']} | {event['company']} ({event.get('ticker','?')})")
    print(f"     Type:    {event.get('event_type')}")
    print(f"     Where:   {event.get('city')}, {event.get('state')}")
    print(f"     Summary: {event.get('summary')}")
    print()

Reviewing 0 events from 30-day window:

